# DyslexiaDetect — EfficientNet-B0 Letter Classifier
Trains a 3-class letter classifier (Normal / Reversal / Corrected) on the Kaggle Dyslexia Handwriting Dataset.

**Before running:** Runtime → Change runtime type → T4 GPU

In [ ]:
# ── Cell 1: Check GPU ──────────────────────────────────────────────────────
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU found. Go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# ── Cell 2: Upload kaggle.json and download dataset ────────────────────────
# You need your kaggle.json API key.
# Get it from: kaggle.com → Your Profile → Settings → API → Create New Token

from google.colab import files
print('Upload your kaggle.json file:')
files.upload()

import os
os.makedirs(os.path.expanduser('~/.config/kaggle'), exist_ok=True)
!cp kaggle.json ~/.config/kaggle/
!chmod 600 ~/.config/kaggle/kaggle.json

print('Downloading dataset from Kaggle...')
!kaggle datasets download -d drizasazanitaisa/dyslexia-handwriting-dataset --force
print('Download complete.')

In [ ]:
# ── Cell 3: Extract dataset (handles password-protected RAR) ───────────────
!apt-get install -y unrar > /dev/null 2>&1
print('unrar installed.')

import zipfile, os

# Step 1: Unzip outer zip
print('Extracting zip...')
with zipfile.ZipFile('dyslexia-handwriting-dataset.zip', 'r') as z:
    print('Zip contents:', z.namelist())
    z.extractall('.')

# Step 2: Extract password-protected RAR
print('Extracting RAR with password...')
!unrar x -pWanAsy321 'Dataset Dyslexia_Password WanAsy321.rar' ./dataset/ -y > /dev/null
print('Extraction complete.')

# Step 3: Find the Gambo folder (Train/Test structure)
dataset_root = None
for root, dirs, _ in os.walk('./dataset'):
    if 'Train' in dirs and 'Test' in dirs:
        dataset_root = root
        break

print(f'Dataset root: {dataset_root}')
print()

# Count images per class
for split in ['Train', 'Test']:
    for cls in ['Normal', 'Reversal', 'Corrected']:
        path = os.path.join(dataset_root, split, cls)
        count = len([f for f in os.listdir(path) if f.lower().endswith('.png')])
        print(f'  {split}/{cls}: {count:,} images')

In [ ]:
# ── Cell 4: Config ─────────────────────────────────────────────────────────
IMG_SIZE       = 224
BATCH_SIZE     = 64
EPOCHS_FREEZE  = 5    # Phase 1: head only, base frozen
EPOCHS_FULL    = 5    # Phase 2: unfreeze top layers
LR_HEAD        = 1e-3
LR_FULL        = 1e-4
NUM_CLASSES    = 3

TRAIN_DIR = os.path.join(dataset_root, 'Train')
TEST_DIR  = os.path.join(dataset_root, 'Test')

print(f'Train dir: {TRAIN_DIR}')
print(f'Test  dir: {TEST_DIR}')

In [ ]:
# ── Cell 5: Data transforms and loaders ───────────────────────────────────
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, WeightedRandomSampler
from collections import Counter

train_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),   # images are grayscale — convert to 3-ch for EfficientNet
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(10),                  # slight rotation — real children tilt their writing
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    # NOTE: NO horizontal flip — flipping Normal creates artificial Reversals
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),    # ImageNet mean/std
])

val_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transforms)
val_dataset   = datasets.ImageFolder(TEST_DIR,  transform=val_transforms)

# Weighted sampler: balances class imbalance (Corrected >> Normal/Reversal)
class_counts = Counter(train_dataset.targets)
sample_weights = [1.0 / class_counts[t] for t in train_dataset.targets]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'Train: {len(train_dataset):,} images')
print(f'Val:   {len(val_dataset):,} images')
print(f'Class mapping: {train_dataset.class_to_idx}')

In [ ]:
# ── Cell 6: Build EfficientNet-B0 model ───────────────────────────────────
import torch.nn as nn

model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)

# Freeze all base layers — only train the new head in Phase 1
for param in model.parameters():
    param.requires_grad = False

# Replace the classifier head with a 3-class output
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.3, inplace=True),
    nn.Linear(in_features, NUM_CLASSES),
)

model = model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable:,} / {total:,}')
print('Classifier head:', model.classifier)

In [ ]:
# ── Cell 7: Training helper functions ─────────────────────────────────────
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    return total_loss / total, correct / total


def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * imgs.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += imgs.size(0)
    return total_loss / total, correct / total

print('Helper functions defined.')

In [ ]:
# ── Cell 8: Phase 1 — Train head only (base frozen) ───────────────────────
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=LR_HEAD
)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

best_val_acc = 0.0
print('Phase 1: Training classifier head (base layers frozen)')
print('-' * 60)

for epoch in range(EPOCHS_FREEZE):
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion)
    vl_loss, vl_acc = eval_epoch(model,   val_loader,  criterion)
    scheduler.step()

    saved = ''
    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(model.state_dict(), 'best_model.pth')
        saved = '  <- saved'

    print(f'Epoch {epoch+1:2d}/{EPOCHS_FREEZE}  '
          f'train loss {tr_loss:.4f} acc {tr_acc:.3f}  '
          f'val loss {vl_loss:.4f} acc {vl_acc:.3f}{saved}')

print(f'\nBest val accuracy (phase 1): {best_val_acc:.3f}')

In [ ]:
# ── Cell 9: Phase 2 — Unfreeze top layers and fine-tune ───────────────────
# Unfreeze the last two feature blocks + classifier
for name, param in model.named_parameters():
    if any(x in name for x in ['features.7', 'features.8', 'classifier']):
        param.requires_grad = True

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=LR_FULL
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_FULL)

print('Phase 2: Fine-tuning top layers')
print('-' * 60)

for epoch in range(EPOCHS_FULL):
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion)
    vl_loss, vl_acc = eval_epoch(model,   val_loader,  criterion)
    scheduler.step()

    saved = ''
    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(model.state_dict(), 'best_model.pth')
        saved = '  <- saved'

    print(f'Epoch {epoch+1:2d}/{EPOCHS_FULL}  '
          f'train loss {tr_loss:.4f} acc {tr_acc:.3f}  '
          f'val loss {vl_loss:.4f} acc {vl_acc:.3f}{saved}')

print(f'\nBest val accuracy overall: {best_val_acc:.3f}')

In [ ]:
# ── Cell 10: Evaluate — confusion matrix & classification report ───────────
!pip install seaborn -q
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

model.load_state_dict(torch.load('best_model.pth'))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        preds = model(imgs.to(device)).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

idx_to_class = {v: k for k, v in val_dataset.class_to_idx.items()}
class_names  = [idx_to_class[i] for i in range(NUM_CLASSES)]

print(classification_report(all_labels, all_preds, target_names=class_names))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix')
plt.ylabel('True')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=100)
plt.show()

In [ ]:
# ── Cell 11: Export to ONNX ────────────────────────────────────────────────
import json

model.load_state_dict(torch.load('best_model.pth'))
model.eval()

dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)

torch.onnx.export(
    model, dummy,
    'letter_classifier.onnx',
    export_params=True,
    opset_version=12,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input':  {0: 'batch_size'},
                  'output': {0: 'batch_size'}},
)
print('Exported: letter_classifier.onnx')

# Save class index mapping so the backend knows which index = which class
class_mapping = {str(v): k for k, v in val_dataset.class_to_idx.items()}
with open('class_mapping.json', 'w') as f:
    json.dump(class_mapping, f, indent=2)
print('Saved: class_mapping.json')
print('Mapping:', class_mapping)

# Quick sanity check
!pip install onnxruntime -q
import onnxruntime as ort
import numpy as np

sess = ort.InferenceSession('letter_classifier.onnx')
test_in = np.random.randn(1, 3, IMG_SIZE, IMG_SIZE).astype(np.float32)
out = sess.run(None, {'input': test_in})[0]
print(f'ONNX output shape: {out.shape}  -> OK')

In [ ]:
# ── Cell 12: Download output files ────────────────────────────────────────
# This will download 3 files to your computer.
# Put letter_classifier.onnx and class_mapping.json in:  backend/models/

from google.colab import files
files.download('letter_classifier.onnx')
files.download('class_mapping.json')
files.download('confusion_matrix.png')
print('Done! Place letter_classifier.onnx and class_mapping.json in backend/models/')